### 1. Φόρτωση Καθαρών Περασμάτων & Προετοιμασία Μεταβλητών

In [ ]:
# Φόρτωση του clean_results.csv, φιλτράρισμα των καθαρών περασμάτων (Is_Clean_Pass == True)
# και δημιουργία ετικετών για τη Διαδρομή (Αριστερή / Δεξιά) και τη Φάση Αγώνα (Qualifying vs Eliminations).

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

PROCESSED_DIR = os.path.join("..", "data", "processed")
FIGURES_DIR = os.path.join("..", "outputs", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

results = pd.read_csv(os.path.join(PROCESSED_DIR, "clean_results.csv"))
clean = results[results['Is_Clean_Pass'] == True].copy()

# Αντιστοίχιση ονομάτων Διαδρομής (1 = Αριστερή, 2 = Δεξιά) και Φάσης Αγώνα
clean['Διαδρομή_Όνομα'] = clean['Διαδρομή'].map({1: 'Αριστερή (Lane 1)', 2: 'Δεξιά (Lane 2)'})
# Αν η ΠεριγραφήΚατάταξης == 'Με Συνολικό Χρόνο' τότε είμαστε σε κόντρες (μετράει και το reaction time), αλλιώς είμαστε σε προκριματικά ή δοκιμαστικά (ΠεριγραφήΚατάταξης == 'Με χρόνο τερματισμού' ή 'Με διαφορά δηλωτού χρόνου')
clean['Φάση_Αγώνα'] = np.where(
    clean['ΠεριγραφήΚατάταξης'] == 'Με Συνολικό Χρόνο',
    'Eliminations (Κόντρες)',
    'Qualifying / Δοκιμαστικά'
)

clean_400 = clean[(clean['Απόσταση_Κατηγορίας_m'] == 400) & (clean['Χρόνος200_sec'] > 0)].copy()
clean_200 = clean[clean['Απόσταση_Κατηγορίας_m'] == 200].copy()

print(f"Σύνολο Καθαρών Περασμάτων: {len(clean)}")
print(f" -> Καθαρά Περάσματα 400m (με πλήρη ενδιάμεσα splits): {len(clean_400)}")
print(f" -> Καθαρά Περάσματα 200m (<200>): {len(clean_200)}")

Σύνολο Καθαρών Περασμάτων: 1782
 -> Καθαρά Περάσματα 400m (με πλήρη ενδιάμεσα splits): 1594
 -> Καθαρά Περάσματα 200m (<200>): 187


### 2. Περιγραφικά Στατιστικά Ενδιάμεσων Χρόνων (Splits 18m, 200m, Back-Half, 400m)

In [4]:
# Υπολογισμός βασικών περιγραφικών στατιστικών (Count, Mean, Median, Std, Min, Max)
# για όλα τα τμήματα της πίστας 400m (18m/60ft, 200m, Back-Half 200-400m, 400m ET) ανά Τύπο Οχήματος.

def calc_stats(series):
    return pd.Series({
        'Count (Πλήθος)': series.count(),
        'Mean (Μέσος Όρος)': series.mean(),
        'Median (Διάμεσος)': series.median(),
        'Std (Τυπ. Απόκλιση)': series.std(),
        'Min (Ελάχιστο)': series.min(),
        'Max (Μέγιστο)': series.max()
    }).round(3)

cars_400 = clean_400[clean_400['ΤύποςΟχήματος'] == 'Αυτοκίνητο']
bikes_400 = clean_400[clean_400['ΤύποςΟχήματος'] == 'Μηχανή']

splits_stats = pd.DataFrame({
    'Αυτοκίνητο 18m (s)': calc_stats(cars_400['Χρόνος18_sec']),
    'Μηχανή 18m (s)': calc_stats(bikes_400['Χρόνος18_sec']),
    'Αυτοκίνητο 200m (s)': calc_stats(cars_400['Χρόνος200_sec']),
    'Μηχανή 200m (s)': calc_stats(bikes_400['Χρόνος200_sec']),
    'Αυτοκίνητο BackHalf 200-400m (s)': calc_stats(cars_400['BackHalf_200_400_sec']),
    'Μηχανή BackHalf 200-400m (s)': calc_stats(bikes_400['BackHalf_200_400_sec']),
    'Αυτοκίνητο 400m ET (s)': calc_stats(cars_400['Χρόνος400_sec']),
    'Μηχανή 400m ET (s)': calc_stats(bikes_400['Χρόνος400_sec'])
})

display(splits_stats)

,Αυτοκίνητο 18m (s),Μηχανή 18m (s),Αυτοκίνητο 200m (s),Μηχανή 200m (s),Αυτοκίνητο BackHalf 200-400m (s),Μηχανή BackHalf 200-400m (s),Αυτοκίνητο 400m ET (s),Μηχανή 400m ET (s)
Count (Πλήθος),569.000,1025.000,569.000,1025.000,569.000,1025.000,569.000,1025.000
Mean (Μέσος Όρος),1.855,1.623,7.749,6.491,4.229,3.684,11.978,10.175
Median (Διάμεσος),1.775,1.596,7.513,6.195,4.033,3.384,11.578,9.548
Std (Τυπ. Απόκλιση),0.302,0.197,1.139,0.780,0.824,0.795,1.799,1.502
Min (Ελάχιστο),1.333,1.196,5.249,5.230,2.755,2.882,8.004,8.282
Max (Μέγιστο),3.334,3.040,13.992,12.536,8.331,7.677,19.831,19.969


### 3. Συσχέτιση & Παλινδρόμηση 18m (60ft) vs 400m ET
- #### Η κλίση δεν είναι αιτιότητα. Οι γρήγορες εκκινήσεις συνήθως συνοδεύουν γρηγορότερα οχήματα, οπότε μέρος του "κέρδους" οφείλεται στο ότι ένα καλύτερο όχημα είναι γρήγορο παντού. Το 0.1 s στα 18m δεν σημαίνει απαραίτητα ότι αν βελτιώσεις μόνο την εκκίνηση θα κερδίσεις τόσο.
- #### Το Pearson r (200m vs 400m) είναι σχεδόν εγγυημένα υψηλό, γιατί το 200m περιέχεται στο 400m, άρα δεν είναι ιδιαίτερα πληροφοριακό.

In [ ]:
# Υπολογισμός συσχέτισης (Pearson r & Spearman r) και γραμμικής παλινδρόμησης (scipy.stats.linregress)
# μεταξύ του χρόνου εκκίνησης 18m (60ft) και του τελικού χρόνου 400m ET.
# Ποσοτικοποιεί πόσο χρόνο κερδίζει ένα όχημα στα 400m για κάθε 0.1s βελτίωσης στα πρώτα 18 μέτρα.

reg_rows = []
for vtype, sub in [('Αυτοκίνητο', cars_400), ('Μηχανή', bikes_400)]:
    slope_18, intercept_18, r_18, p_18, _ = stats.linregress(sub['Χρόνος18_sec'], sub['Χρόνος400_sec'])
    r_sp_18, _ = stats.spearmanr(sub['Χρόνος18_sec'], sub['Χρόνος400_sec'])
    slope_200, _, r_200, _, _ = stats.linregress(sub['Χρόνος200_sec'], sub['Χρόνος400_sec'])
    
    reg_rows.append({
        'Τύπος Οχήματος': vtype,
        'Πλήθος (n)': len(sub),
        'Pearson r (18m vs 400m)': round(r_18, 3),
        'Spearman r (18m vs 400m)': round(r_sp_18, 3),

        # Η κλίση (slope_18) δείχνει πόσα δευτερόλεπτα αλλάζει ο χρόνος 400m για κάθε 1 s αλλαγή στα 18m.
        # Επειδή το 1 s στα 18m είναι μη ρεαλιστικά μεγάλη διαφορά, πολλαπλασιάζουμε επί 0.1 ώστε το αποτέλεσμα
        # να αναφέρεται σε μια ρεαλιστική βελτίωση 0.1 s (η σχέση είναι γραμμική, άρα κέρδος = κλίση × Δx).
        'Κέρδος 400m ανά -0.1s στα 18m (s)': round(slope_18 * 0.1, 3),
        
        'Pearson r (200m vs 400m)': round(r_200, 3),
        'p-value (18m vs 400m)': f"{p_18:.2e}"
    })

reg_df = pd.DataFrame(reg_rows)
display(reg_df)

,Τύπος Οχήματος,Πλήθος (n),Pearson r (18m vs 400m),Spearman r (18m vs 400m),Κέρδος 400m ανά -0.1s στα 18m (s),Pearson r (200m vs 400m),p-value (18m vs 400m)
0,Αυτοκίνητο,569,0.709,0.631,0.422,0.941,3.25e-88
1,Μηχανή,1025,0.489,0.502,0.374,0.953,8.76e-63


### **Τι δείχνει ο έλεγχος συσχέτισης και γραμμικής παλινδρόμησης (18m & 200m vs 400m):**

- ##### **Αυτοκίνητα:** Ο χρόνος στα πρώτα 18m (60ft) σχετίζεται ισχυρά και στατιστικά σημαντικά θετικά με τον τελικό χρόνο στα 400m (Pearson r = 0.709, Spearman r = 0.631, p < 0.0001). Ο συντελεστής προσδιορισμού R² = 0.503 σημαίνει ότι περίπου το 50% της διακύμανσης του χρόνου 400m συνδέεται με τον χρόνο 18m. Η κλίση της ευθείας παλινδρόμησης μας δείχνει ότι: **Κάθε 0.1 s ταχύτερη εκκίνηση συνδέεται κατά μέσο όρο με 0.422 s ταχύτερο τερματισμό**.
- ##### **Μηχανές:** Η συσχέτιση του 18αριού με τον τελικό χρόνο 400m είναι μέτρια και στατιστικά σημαντική (Pearson r = 0.489, Spearman r = 0.502, p < 0.0001, R² ≈ 0.24). **Κάθε 0.1 s ταχύτερη εκκίνηση συνδέεται κατά μέσο όρο με 0.374 s ταχύτερο τερματισμό**. Η εκκίνηση φαίνεται να προβλέπει λιγότερο τον τελικό χρόνο στις μηχανές απ' ό,τι στα αυτοκίνητα.
- ##### **Σημείο των 200m:** Η συσχέτιση με τον τελικό χρόνο 400m είναι πολύ υψηλή στα αυτοκίνητα (r = 0.941, R² = 0.885) και στις μηχανές (r = 0.953, R² = 0.908), επιβεβαιώνοντας ότι στα 200 μέτρα η επίδοση του περάσματος έχει ήδη κριθεί σε ποσοστό ≈90%. Είναι αναμενόμενο, αφού τα 200m αποτελούν το μισό της διαδρομής.
- ##### **Προσοχή στην ερμηνεία:** Πρόκειται για συσχέτιση και όχι αποδεδειγμένη αιτιότητα. Ένα όχημα που ξεκινά γρήγορα είναι συνήθως γρήγορο και στη συνέχεια, και οι κατηγορίες διαφέρουν σε ισχύ, οπότε μέρος της σχέσης οφείλεται σε αυτό.

### 4. Ανάλυση Χρόνων Αντίδρασης (Net_RT_sec) & Red Lights (Qualifying vs Eliminations)
- #### Εξαιρέθηκαν περάσματα με καθαρό χρόνο αντίδρασης `Net RT > 1.5s`, καθώς αντιπροσωπεύουν μηχανική καθυστέρηση turbo / εμπλοκής DSG (spool-up / launch control) ή ηθελημένη αναμονή στα δοκιμαστικά, και όχι ανθρώπινο χρόνο αντίδρασης

In [ ]:
# Ανάλυση του Καθαρού Χρόνου Αντίδρασης (Net_RT_sec = ΧρόνοςΑντίδρασης_sec - 0.400)
# και του ποσοστού Άκυρων Εκκινήσεων (Red Lights / Εγκυρη == -1) ανάμεσα στα Προκριματικά
# (Qualifying / Δοκιμαστικά) και τις Κόντρες (Eliminations).

# 1. Στατιστικά Καθαρού Χρόνου Αντίδρασης (έγκυρες αντιδράσεις 0.000s έως 1.500s)
    # ΔΕΝ ΠΡΕΠΕΙ να συμπεριλάβω και τα DSG/Turbo αυτοκίνητα που χρειάζονται παραπάνω δευτερόλεπτα πριν πάρουν εκκίνηση για να χτίσουν πίεση γιατί ΧΑΝΕΤΑΙ ΤΟ ΝΟΗΜΑ της ανάλυσης!
rt_clean = clean[clean['Net_RT_sec'].between(0.0, 1.5)].copy()

rt_summary = (
    rt_clean.groupby(['Φάση_Αγώνα', 'ΤύποςΟχήματος'])['Net_RT_sec']
    .agg(Count='count', Mean='mean', Median='median', Std='std', Min='min', Max='max')
    .round(3)
    .reset_index()
)
print("1. Καθαρός Χρόνος Αντίδρασης (Net_RT_sec) ανά Φάση Αγώνα & Τύπο Οχήματος:")
display(rt_summary)

# 2. Συχνότητα Άκυρων Εκκινήσεων (Red Lights: Εγκυρη == -1) ανά Φάση Αγώνα
champ_runs = results[(results['ΠΙΣΤΑ'] != 'ΔΟΚΙΜΕΣ') & (results['ααΣυμμετοχής'] > 0)].copy()
champ_runs['Φάση_Αγώνα'] = np.where(
    champ_runs['ΠεριγραφήΚατάταξης'] == 'Με Συνολικό Χρόνο',
    'Eliminations (Κόντρες)',
    'Qualifying / Δοκιμαστικά'
)

rl_summary = champ_runs.groupby('Φάση_Αγώνα').agg(
    Σύνολο_Περασμάτων=('ααΕπίδοσης', 'count'),
    Red_Lights=('Εγκυρη', lambda s: (s == -1).sum())
).reset_index()
rl_summary['Ποσοστό Red Light (%)'] = (
    rl_summary['Red_Lights'] / rl_summary['Σύνολο_Περασμάτων'] * 100
).round(2)

print("\n2. Ποσοστό Άκυρων Εκκινήσεων (Red Lights) ανά Φάση Αγώνα:")
display(rl_summary)

1. Καθαρός Χρόνος Αντίδρασης (Net_RT_sec) ανά Φάση Αγώνα & Τύπο Οχήματος:


,Φάση_Αγώνα,ΤύποςΟχήματος,Count,Mean,Median,Std,Min,Max
0,Eliminations (Κόντρες),Αυτοκίνητο,112,0.518,0.422,0.350,0.019,1.495
1,Eliminations (Κόντρες),Μηχανή,188,0.241,0.183,0.220,0.002,1.428
2,Qualifying / Δοκιμαστικά,Αυτοκίνητο,246,0.652,0.602,0.347,0.023,1.490
3,Qualifying / Δοκιμαστικά,Μηχανή,866,0.451,0.344,0.341,0.000,1.493



2. Ποσοστό Άκυρων Εκκινήσεων (Red Lights) ανά Φάση Αγώνα:


,Φάση_Αγώνα,Σύνολο_Περασμάτων,Red_Lights,Ποσοστό Red Light (%)
0,Eliminations (Κόντρες),534,29,5.43
1,Qualifying / Δοκιμαστικά,1677,20,1.19


Τι δείχνει η ανάλυση των Χρόνων Αντίδρασης (Net_RT_sec) & Red Lights (`Qualifying` vs `Eliminations`): 
- Στις Μηχανές: Ο διάμεσος καθαρός χρόνος αντίδρασης πέφτει σχεδόν στο μισό στις Κόντρες (Eliminations) ($0.183\text{ s}$) σε σχέση με τα Προκριματικά ($0.344\text{ s}$).
- Στα Αυτοκίνητα: Ο διάμεσος καθαρός χρόνος αντίδρασης πέφτει από $0.602\text{ s}$ στα Προκριματικά σε $0.422\text{ s}$ στις Κόντρες.
- Red Lights (Εγκυρη == -1): Επειδή στις Κόντρες μετράει ο συνολικός χρόνος (Αντίδραση + ET), οι οδηγοί πιέζουν το φανάρι στα όρια, με αποτέλεσμα το ποσοστό των Red Lights να τετραπλασιάζεται από $1.19\%$ στα Προκριματικά ($20 / 1.677$) σε $5.43\%$ στις Κόντρες ($29 / 534$)!

### 5. Έλεγχος Ισοτιμίας Διαδρομών (Lane Bias Audit: Αριστερή vs Δεξιά Διαδρομή)